# 01 — Core Numeric & Date Columns (Team 1)

## Columns Assigned
`Fiyat`, `Yıl`, `Kilometre`, `İlan Tarihi`, `Ortalama Kasko`,
`Ortalama Trafik Sigortası`, `Üretim Yılı (İlk/Son)`,
`Silindir Sayısı`, `Koltuk Sayısı`, `Bagaj Hacmi`, `Yakıt Deposu`

## What is Expected as Output
- `Fiyat`: strip "TL" and dots → integer (e.g. "2.425.000TL" → 2425000)
- `Yıl`: integer, nulls filled with median
- `Kilometre`: strip " km" and dots → integer, nulls filled with median
- `İlan Tarihi`: convert to number of days since the earliest listing date → integer
- `Ortalama Kasko`: strip "TL" and dots → integer, nulls filled with median
- `Ortalama Trafik Sigortası`: strip "TL" and dots → integer, nulls filled with median
- `Üretim Yılı (İlk/Son)`: extract the start year from ranges like "2014 - 2017" → integer 2014, nulls filled with median
- `Silindir Sayısı`: already mostly numeric, cast to integer, nulls filled with median
- `Koltuk Sayısı`: already mostly numeric, cast to integer, nulls filled with median
- `Bagaj Hacmi`: strip " lt" → integer, nulls filled with median
- `Yakıt Deposu`: strip " lt" → integer, nulls filled with median
- No nulls remaining, all columns integer or float, no string columns

## Output variable: df_team1

---

> ## ⚠️ READ THIS BEFORE YOU WRITE A SINGLE LINE OF CODE ⚠️
>
> ### The code below is a PLACEHOLDER — it is NOT the final solution.
>
> The cells in this notebook show **one possible way** to process the assigned columns. You are **not** required to follow this approach. You may completely replace it with your own method if you believe yours is better suited to the data.
>
> ### What IS required — without exception:
>
> **Every code cell must be accompanied by a markdown cell that explains:**
> 1. **What you did** — describe the operation in plain language.
> 2. **Why you did it** — justify the choice. Why did you think this specific approach would work? What problem does it solve? Why this method and not another?
>
> A notebook that only contains code — or markdown cells that just repeat the code in words — will **not** be accepted.
>
> **Not acceptable:** `## Step 2` or `# fill nulls`
>
> **Acceptable:** *"We imputed missing values in `Kilometre` with the median rather than the mean because the column is heavily right-skewed (a small number of very high-mileage listings would pull the mean upward, misrepresenting the majority of vehicles). The median is robust to those outliers and better reflects a typical listing."*
>
> Document every decision. If you tried an approach and abandoned it, explain why.

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

RAW_URL = "https://raw.githubusercontent.com/MamoMGD1/ISE302-DataMining-GroupProject/main/data/raw_dataset.csv"
df_full = pd.read_csv(RAW_URL)
print(f"Loaded dataset: {df_full.shape}")
df_full.head(3)

In [ ]:
MY_COLUMNS = [
    'Fiyat', 'Yıl', 'Kilometre', 'İlan Tarihi',
    'Ortalama Kasko', 'Ortalama Trafik Sigortası',
    'Üretim Yılı (İlk/Son)', 'Silindir Sayısı',
    'Koltuk Sayısı', 'Bagaj Hacmi', 'Yakıt Deposu'
]
df = df_full[MY_COLUMNS].copy()

## Step 1 — Null Analysis

In [ ]:
# Print null counts and percentage for each column
null_counts = df.isnull().sum()
null_pct = (null_counts / len(df) * 100).round(2)
print(pd.DataFrame({'null_count': null_counts, 'null_pct': null_pct}))

In [ ]:
# Clean Fiyat: strip 'TL' and dot thousands separators, then cast to int
df['Fiyat'] = (
    df['Fiyat']
    .astype(str)
    .str.replace('TL', '', regex=False)
    .str.replace('.', '', regex=False)
    .str.strip()
    .astype(int)
)
print(f"Fiyat nulls: {df['Fiyat'].isnull().sum()}")

In [ ]:
# Clean Yıl: cast to numeric, fill nulls with median, then cast to int
df['Yıl'] = pd.to_numeric(df['Yıl'], errors='coerce')
df['Yıl'] = df['Yıl'].fillna(df['Yıl'].median()).astype(int)
print(f"Yıl nulls: {df['Yıl'].isnull().sum()}")

In [ ]:
# Clean Kilometre: strip ' km' and dot thousands separators, fill nulls with median, cast to int
df['Kilometre'] = (
    df['Kilometre']
    .astype(str)
    .str.replace(' km', '', regex=False)
    .str.replace('.', '', regex=False)
    .str.strip()
    .replace('nan', np.nan)
)
df['Kilometre'] = pd.to_numeric(df['Kilometre'], errors='coerce')
df['Kilometre'] = df['Kilometre'].fillna(df['Kilometre'].median()).astype(int)
print(f"Kilometre nulls: {df['Kilometre'].isnull().sum()}")

In [ ]:
# Clean İlan Tarihi: parse as datetime, then convert to days since earliest listing date
df['İlan Tarihi'] = pd.to_datetime(df['İlan Tarihi'], dayfirst=True, errors='coerce')
min_date = df['İlan Tarihi'].min()
df['İlan Tarihi'] = (df['İlan Tarihi'] - min_date).dt.days
# Fill any remaining nulls (unparseable dates) with median
df['İlan Tarihi'] = df['İlan Tarihi'].fillna(df['İlan Tarihi'].median()).astype(int)
print(f"İlan Tarihi nulls: {df['İlan Tarihi'].isnull().sum()}")

In [ ]:
# Clean insurance columns: strip 'TL' and dot thousands separators → integer
for col in ['Ortalama Kasko', 'Ortalama Trafik Sigortası']:
    df[col] = (
        df[col].astype(str)
        .str.replace('TL', '', regex=False)
        .str.replace('.', '', regex=False)
        .str.strip()
        .replace('nan', np.nan)
    )
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df[col] = df[col].fillna(df[col].median()).astype(int)
    print(f"{col} nulls: {df[col].isnull().sum()}")

In [ ]:
# Clean Üretim Yılı (İlk/Son): extract start year from ranges like '2014 - 2017'
df['Üretim Yılı (İlk/Son)'] = (
    df['Üretim Yılı (İlk/Son)']
    .astype(str)
    .str.extract(r'(\d{4})')[0]
)
df['Üretim Yılı (İlk/Son)'] = pd.to_numeric(df['Üretim Yılı (İlk/Son)'], errors='coerce')
df['Üretim Yılı (İlk/Son)'] = df['Üretim Yılı (İlk/Son)'].fillna(df['Üretim Yılı (İlk/Son)'].median()).astype(int)
print(f"Üretim Yılı (İlk/Son) nulls: {df['Üretim Yılı (İlk/Son)'].isnull().sum()}")

In [ ]:
# Clean Silindir Sayısı: cast to numeric, fill nulls with median, cast to int
df['Silindir Sayısı'] = pd.to_numeric(df['Silindir Sayısı'], errors='coerce')
df['Silindir Sayısı'] = df['Silindir Sayısı'].fillna(df['Silindir Sayısı'].median()).astype(int)
print(f"Silindir Sayısı nulls: {df['Silindir Sayısı'].isnull().sum()}")

In [ ]:
# Clean Koltuk Sayısı: cast to numeric, fill nulls with median, cast to int
df['Koltuk Sayısı'] = pd.to_numeric(df['Koltuk Sayısı'], errors='coerce')
df['Koltuk Sayısı'] = df['Koltuk Sayısı'].fillna(df['Koltuk Sayısı'].median()).astype(int)
print(f"Koltuk Sayısı nulls: {df['Koltuk Sayısı'].isnull().sum()}")

In [ ]:
# Clean Bagaj Hacmi: strip ' lt', fill nulls with median, cast to int
df['Bagaj Hacmi'] = (
    df['Bagaj Hacmi'].astype(str)
    .str.replace(' lt', '', regex=False)
    .str.strip()
    .replace('nan', np.nan)
)
df['Bagaj Hacmi'] = pd.to_numeric(df['Bagaj Hacmi'], errors='coerce')
df['Bagaj Hacmi'] = df['Bagaj Hacmi'].fillna(df['Bagaj Hacmi'].median()).astype(int)
print(f"Bagaj Hacmi nulls: {df['Bagaj Hacmi'].isnull().sum()}")

In [ ]:
# Clean Yakıt Deposu: strip ' lt', fill nulls with median, cast to int
df['Yakıt Deposu'] = (
    df['Yakıt Deposu'].astype(str)
    .str.replace(' lt', '', regex=False)
    .str.strip()
    .replace('nan', np.nan)
)
df['Yakıt Deposu'] = pd.to_numeric(df['Yakıt Deposu'], errors='coerce')
df['Yakıt Deposu'] = df['Yakıt Deposu'].fillna(df['Yakıt Deposu'].median()).astype(int)
print(f"Yakıt Deposu nulls: {df['Yakıt Deposu'].isnull().sum()}")

In [ ]:
df_team1 = df.copy()
assert df_team1.isnull().sum().sum() == 0, "Nulls remain!"
assert df_team1.select_dtypes(exclude='number').shape[1] == 0, "Non-numeric columns remain!"
print(f"✅ Team 1 output ready. Shape: {df_team1.shape}")
print(df_team1.dtypes)